# Beehive Bee Detector — Colab Training

Trains the worker/drone/queen/pollen_bearer YOLOv8 detector for the `beehive_monitor` project on a free Colab GPU, then hands you a `best.pt` to copy back to your Ubuntu box (training is the only step that benefits from a GPU — the live pipeline runs fine on CPU).

**Before running this notebook:**
1. On your Ubuntu box, run `python train/merge_datasets.py --sources ... --out train/data` to build the merged dataset (see TRAINING_PLAN.md).
2. Zip it: `cd beehive_monitor && zip -r train_data.zip train/data`
3. Have `train_data.zip` ready to upload in the cell below. (The dataset itself is gitignored due to size — it doesn't travel through the repo, just this direct upload.)

**Set the runtime first:** Runtime → Change runtime type → T4 GPU, before running any cells.

In [ ]:
# Confirm you actually got a GPU runtime -- if this errors, go set
# Runtime > Change runtime type > T4 GPU, then re-run from the top.
!nvidia-smi

In [ ]:
!pip install -q ultralytics

In [ ]:
# Upload train_data.zip (the merged dataset you built with merge_datasets.py)
from google.colab import files
import zipfile

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(".")
print("Extracted. Contents of train/data:")
!ls -R train/data | head -30

In [ ]:
# merge_datasets.py wrote an absolute path from YOUR Ubuntu box into data.yaml
# (the 'path:' field) -- that path doesn't exist here, so rewrite it to
# Colab's local path before training, or Ultralytics will fail to find images.
import os
import yaml

data_yaml_path = "train/data/data.yaml"
with open(data_yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg["path"] = os.path.abspath("train/data")
with open(data_yaml_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)
print(cfg)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="train/data/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    project="runs",
    name="bee_detector",
    patience=20,
)
# ~10-20 minutes on a free T4 for a dataset in the low thousands of images.
# If Colab disconnects you for being idle, just re-run from the top -- training
# doesn't resume mid-run on the free tier, so don't walk away for hours.

In [ ]:
metrics = model.val()
print(metrics)

In [ ]:
# Downloads to your browser's normal download folder.
from google.colab import files

files.download("runs/bee_detector/weights/best.pt")

## Next steps back on your Ubuntu box

1. Move the downloaded `best.pt` into `beehive_monitor/models/`.
2. In `config.yaml`, set `model_path: "models/best.pt"`.
3. Run `python -m src.calibrate --config config.yaml` to set your counting line, then `python -m src.pipeline --config config.yaml` to start watching the hive.
4. Evaluate against a real clip per TRAINING_PLAN.md step 3 before trusting the counts.